In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator,JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
env, _ = setup_cart_pole_env()

## Cart dynamics:

In [ ]:
for init_omega in jnp.arange(-1, 1, 0.1):
    init_obs = jnp.array([0, 0, 0., init_omega])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=env.env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

In [ ]:
for init_v in jnp.arange(-1, 0, 0.1):
    init_obs = jnp.array([0, init_v, 0., 0])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=env.env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

In [ ]:
for init_theta in jnp.arange(-1, 1, 0.1):
    for init_omega in jnp.arange(-1, 1, 0.1):
        init_obs = jnp.array([0, 0, init_theta, init_omega])
        init_state = env.generate_state_from_observation(init_obs, env.env_properties)
        
        actions = jnp.ones((100,1)) * 0.5
            
        observations, states, last_state = env.sim_ahead(
            init_state,
            actions=actions,
            env_properties=env.env_properties,
            obs_stepsize=env.tau,
            action_stepsize=env.tau,
        )
        plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

In [ ]:
def recursive_feasibility_cart(d_k, v_k, u_k, tau, d_max, v_max, u_max, static_params):
    valid_deflection = (jnp.abs(d_k) <= d_max)
    valid_velocity = (jnp.abs(v_k) <= v_max)
    valid_action = (jnp.abs(u_k) <= u_max)
    valid_base = jnp.logical_and(valid_deflection, valid_velocity)
    valid_base = jnp.logical_and(valid_action, valid_base)

    valid_constr_d_lower = (v_k >= 1 / tau * (-d_max - d_k))
    valid_constr_d_upper = (v_k <= 1 / tau * (d_max - d_k))
    valid_constr_d = jnp.logical_and(valid_constr_d_lower, valid_constr_d_upper)

    valid_constr_action_v_lower = (u_k >= (static_params.m_c + static_params.m_p) / tau * (-v_max -v_k))
    valid_constr_action_v_upper = (u_k <= (static_params.m_c + static_params.m_p) / tau * (v_max -v_k))
    valid_constr_action_v = jnp.logical_and(valid_constr_action_v_lower, valid_constr_action_v_upper)

    # valid_constr_action_d_lower = (
    #     u_k >= ((static_params.m_c + static_params.m_p) / tau**2 * (-d_max - d_k - tau * v_k) 
    #     - (static_params.m_c + static_params.m_p) / tau * v_k)
    # )      
    # valid_constr_action_d_upper = (
    #     u_k <= ((static_params.m_c + static_params.m_p) / tau**2 * (d_max - d_k - tau * v_k) 
    #     - (static_params.m_c + static_params.m_p) / tau * v_k)
    # )      
    # valid_constr_action_d = jnp.logical_and(valid_constr_action_d_lower, valid_constr_action_d_upper)

    valid = jnp.logical_and(valid_base, valid_constr_d)
    valid = jnp.logical_and(valid, valid_constr_action_v)
    # valid = jnp.logical_and(valid, valid_constr_action_d)

    # valid = valid_base


    recursive_state_constr = v_k**2 * (static_params.m_c + static_params.m_p) / (2 * u_max) <= d_max - jnp.sign(d_k * v_k) * jnp.abs(d_k)
    valid = jnp.logical_and(valid, recursive_state_constr)

    def true_fun():
        valid_lower = (u_k >= (static_params.m_c + static_params.m_p) / tau * (-v_k - jnp.sqrt(2*u_max / (static_params.m_c + static_params.m_p) * (d_max - jnp.abs(d_k + tau * v_k)))))
        valid_upper = (u_k <= (static_params.m_c + static_params.m_p) / tau * (-v_k + jnp.sqrt(2*u_max / (static_params.m_c + static_params.m_p) * (d_max - jnp.abs(d_k + tau * v_k)))))
        return jnp.logical_and(valid_lower, valid_upper)
    
    def false_fun():
        valid_lower = (u_k >= (static_params.m_c + static_params.m_p) / tau * (-v_k - jnp.sqrt(2*u_max / (static_params.m_c + static_params.m_p) * (d_max + jnp.abs(d_k + tau * v_k)))))
        valid_upper = (u_k <= (static_params.m_c + static_params.m_p) / tau * (-v_k + jnp.sqrt(2*u_max / (static_params.m_c + static_params.m_p) * (d_max + jnp.abs(d_k + tau * v_k)))))
        return jnp.logical_and(valid_lower, valid_upper)

    v_k_next = v_k + tau * u_k / ((static_params.m_c + static_params.m_p))
    d_k_next = d_k + tau * v_k
    
    recursive_action_constr = jax.lax.cond(jnp.sign(v_k_next) == jnp.sign(d_k_next), true_fun, false_fun)
    valid = jnp.logical_and(valid, recursive_action_constr)
    return valid

In [ ]:
d_max=env.env_properties.physical_normalizations.deflection.max
v_max=env.env_properties.physical_normalizations.velocity.max
u_max=env.env_properties.action_normalizations.force.max

In [ ]:
recursive_feasibility_cart_parameterized = eqx.filter_jit(
    partial(
        recursive_feasibility_cart,
        tau=env.tau,
        d_max=d_max,
        v_max=v_max,
        u_max=u_max,
        static_params=env.env_properties.static_params
    )
)
recursive_feasibility_cart_parameterized

In [ ]:
deflections_ = jnp.linspace(-d_max, d_max, 250)
velocities_ = jnp.linspace(-v_max, v_max, 250)
forces_ = jnp.linspace(-u_max, u_max, 250)
deflections, velocities, forces = jnp.meshgrid(deflections_, velocities_, forces_)

out_bool = jax.vmap(jax.vmap(jax.vmap(recursive_feasibility_cart_parameterized)))(deflections, velocities, forces)

fig, ax = plt.subplots(1,3, figsize=(10,10), sharey=True)

ax[0].imshow(jnp.any(out_bool, axis=-1), cmap="plasma", extent=[-d_max, d_max, v_max, -v_max], interpolation="nearest")
ax[0].set_xlabel(r"$d$")
ax[0].set_ylabel(r"$v$")

ax[1].imshow(jnp.all(out_bool, axis=-1), cmap="plasma", extent=[-d_max, d_max, v_max, -v_max], interpolation="nearest")
ax[1].set_xlabel(r"$d$")

ax[2].imshow(jnp.logical_xor(jnp.any(out_bool,  axis=-1), jnp.all(out_bool,  axis=-1)), cmap="plasma", extent=[-d_max, d_max, v_max, -v_max],  interpolation="nearest")
ax[2].set_xlabel(r"$d$")
# plt.savefig("recursive_feasible_space.png", dpi=400)
fig.tight_layout()
plt.show()

## Pendulum dynamics: